# Saudi Digital Concierge — Data Exploration
## Source 5: Enjoy.sa Events API

**Endpoint:** `https://enjoy.sa/api/v1/odp/events/Get`
**Purpose:** events, dates, times, city, category, and family/gender constraints.

This notebook fetches the API and answers 12 specific questions about the data. It uses
**defensive schema discovery** — column names are located by keyword, so the analysis
adapts to whatever fields the API returns (English or Arabic).

> **Run this on a machine with internet access to `enjoy.sa`.** If the endpoint needs
> query params, an API key, or a specific host header, adjust `PARAMS`/`HEADERS` below.

### Questions answered
1. Number of events · 2. Available fields · 3. Date coverage · 4. City coverage ·
5. Event categories/types · 6. Start/end dates · 7. Start/end times ·
8. Family/male/female restrictions · 9. Missing values · 10. Duplicate events ·
11. Active/future events? · 12. Pagination / record limits

In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
%pip install requests
import requests
from datetime import datetime, timezone

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

URL = "https://enjoy.sa/api/v1/odp/events/Get"
HEADERS = {"Accept": "application/json", "User-Agent": "saudi-digital-concierge/1.0"}
PARAMS = {}          # add query params here if the API requires them
TIMEOUT = 60
from datetime import datetime, timezone

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

URL = "https://enjoy.sa/api/v1/odp/events/Get"
HEADERS = {"Accept": "application/json", "User-Agent": "saudi-digital-concierge/1.0"}
PARAMS = {}          # add query params here if the API requires them
TIMEOUT = 60

ModuleNotFoundError: No module named 'requests'

## 0. Fetch the raw response

We first inspect the *shape* of the response — a bare JSON array, or an object that wraps
the records (e.g. `{"data": [...]}`, `{"items": [...]}`, `{"result": {"events": [...]}}`)
and may carry paging metadata.

In [ ]:
resp = requests.get(URL, headers=HEADERS, params=PARAMS, timeout=TIMEOUT)
print("HTTP", resp.status_code, "| content-type:", resp.headers.get("content-type"))
resp.raise_for_status()
payload = resp.json()

print("Top-level type:", type(payload).__name__)
if isinstance(payload, dict):
    print("Top-level keys:", list(payload.keys()))

In [ ]:
def extract_records(payload):
    """Return the list of event records from whatever envelope the API uses."""
    if isinstance(payload, list):
        return payload, None
    if isinstance(payload, dict):
        # common list-bearing keys, checked in order
        for key in ["data", "items", "results", "result", "events",
                    "records", "value", "Data", "Items", "Result", "Events"]:
            if key in payload and isinstance(payload[key], list):
                return payload[key], payload
        # one level deeper (e.g. {"result": {"events": [...]}})
        for v in payload.values():
            if isinstance(v, dict):
                for key, inner in v.items():
                    if isinstance(inner, list):
                        return inner, payload
        # otherwise the dict itself may be a single record
        return [payload], payload
    return [], payload

records, envelope = extract_records(payload)
print("Extracted", len(records), "record(s).")
if records:
    print("First record keys:", list(records[0].keys()) if isinstance(records[0], dict) else type(records[0]))

In [ ]:
# Flatten nested JSON into a tidy table.
df = pd.json_normalize(records)
print("DataFrame shape:", df.shape)
df.head(3)

## 1. Number of events

In [ ]:
print("Number of records returned:", len(df))
# NOTE: this is the count in THIS response; see Q12 (pagination) for the true total.

## 2. Available fields

In [ ]:
fields = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(t) for t in df.dtypes],
    "non_null": df.notna().sum().values,
    "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
    "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
})
fields

In [ ]:
# Helper: locate columns whose name contains any of the given keywords.
def find_cols(keywords):
    kws = [k.lower() for k in keywords]
    return [c for c in df.columns if any(k in c.lower() for k in kws)]

col_map = {
    "city":      find_cols(["city", "region", "location", "المدينة", "مدينه", "منطقة"]),
    "category":  find_cols(["category", "categories", "type", "tag", "التصنيف", "نوع", "فئة"]),
    "start":     find_cols(["start", "from", "begin", "بداية", "من"]),
    "end":       find_cols(["end", "to", "finish", "نهاية", "الى", "إلى"]),
    "date":      find_cols(["date", "day", "تاريخ", "يوم"]),
    "time":      find_cols(["time", "hour", "وقت", "ساعة"]),
    "gender":    find_cols(["gender", "male", "female", "family", "audience",
                            "جنس", "رجال", "نساء", "عائلات", "عائلي"]),
    "name":      find_cols(["name", "title", "اسم", "عنوان"]),
}
for k, v in col_map.items():
    print(f"{k:10s}: {v}")

## 3. Date coverage
The earliest and latest dates present, across all detected date columns.

In [ ]:
date_cols = sorted(set(col_map["date"] + col_map["start"] + col_map["end"]))
print("Date-like columns:", date_cols)

parsed = {}
for col in date_cols:
    s = pd.to_datetime(df[col], errors="coerce", utc=True)
    if s.notna().any():
        parsed[col] = s
        print(f"  {col:30s} min={s.min()}  max={s.max()}  parsed={s.notna().sum()}/{len(s)}")

if parsed:
    all_dates = pd.concat(parsed.values())
    print("\nOVERALL date coverage:", all_dates.min(), "->", all_dates.max())
else:
    print("No parseable date columns found — inspect the fields table in Q2.")

## 4. City coverage

In [ ]:
if col_map["city"]:
    for col in col_map["city"]:
        vc = df[col].value_counts(dropna=False)
        print(f"=== {col} — {df[col].nunique(dropna=True)} unique ===")
        print(vc.head(30).to_string(), "\n")
else:
    print("No city/location column detected — check Q2 fields.")

## 5. Event categories / types

In [ ]:
if col_map["category"]:
    for col in col_map["category"]:
        print(f"=== {col} — {df[col].nunique(dropna=True)} unique ===")
        # categories may be lists; explode if so
        sample = df[col].dropna()
        if len(sample) and isinstance(sample.iloc[0], list):
            print(df[col].explode().value_counts().head(30).to_string(), "\n")
        else:
            print(df[col].value_counts(dropna=False).head(30).to_string(), "\n")
else:
    print("No category/type column detected — check Q2 fields.")

## 6. Start / end dates &nbsp;·&nbsp; 7. Start / end times

In [ ]:
start_cols = col_map["start"]
end_cols   = col_map["end"]
print("Start columns:", start_cols)
print("End columns:  ", end_cols)

def summarise_datetime(col):
    s = pd.to_datetime(df[col], errors="coerce", utc=True)
    if s.notna().any():
        print(f"  {col}: {s.min()} -> {s.max()} "
              f"(parsed {s.notna().sum()}/{len(s)})")
        # time-of-day distribution
        hours = s.dt.hour.dropna()
        if len(hours):
            print(f"      hour-of-day range: {int(hours.min())}h - {int(hours.max())}h")

print("\n-- Start --")
for col in start_cols: summarise_datetime(col)
print("-- End --")
for col in end_cols: summarise_datetime(col)

# Dedicated time columns (if separate from dates)
time_only = [c for c in col_map["time"] if c not in start_cols + end_cols]
if time_only:
    print("\n-- Standalone time columns --")
    for col in time_only:
        print(f"  {col}: sample ->", df[col].dropna().head(5).tolist())

## 8. Family / male / female restrictions
Audience-restriction fields tell the concierge who an event is open to.

In [ ]:
if col_map["gender"]:
    for col in col_map["gender"]:
        print(f"=== {col} ===")
        sample = df[col].dropna()
        if len(sample) and isinstance(sample.iloc[0], list):
            print(df[col].explode().value_counts(dropna=False).to_string(), "\n")
        else:
            print(df[col].value_counts(dropna=False).to_string(), "\n")
else:
    print("No explicit gender/family/audience column detected.")
    print("Scan the fields table in Q2 for values like 'Family', 'Men', 'Women', "
          "'عائلات', 'رجال', 'نساء'.")

## 9. Missing values

In [ ]:
miss = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(1),
}).sort_values("missing", ascending=False)
print("Columns with any missing values:")
miss[miss["missing"] > 0]

## 10. Duplicate events
Checked two ways: fully identical rows, and duplicates on a likely business key
(id, or name + start).

In [ ]:
print("Fully identical rows:", df.duplicated().sum())

id_cols = find_cols(["id", "uuid", "guid", "code"])
id_cols = [c for c in id_cols if "video" not in c.lower() and "image" not in c.lower()]
print("Candidate id columns:", id_cols)
for col in id_cols[:2]:
    dups = df[col].duplicated().sum()
    print(f"  duplicate {col}: {dups}")

key = col_map["name"][:1] + col_map["start"][:1]
if len(key) >= 1:
    print(f"\nDuplicates on business key {key}:", df.duplicated(subset=key).sum())

## 11. Active / future events?
Compares start/end dates to *now* to see whether the API serves upcoming events or only
a historical dump.

In [ ]:
now = pd.Timestamp.now(tz="UTC")
print("Now:", now)

start_col = col_map["start"][0] if col_map["start"] else (date_cols[0] if date_cols else None)
end_col   = col_map["end"][0] if col_map["end"] else start_col

if start_col:
    s = pd.to_datetime(df[start_col], errors="coerce", utc=True)
    e = pd.to_datetime(df[end_col], errors="coerce", utc=True) if end_col else s
    future  = (s > now).sum()
    ongoing = ((s <= now) & (e >= now)).sum()
    past    = (e < now).sum()
    print(f"Using start='{start_col}', end='{end_col}'")
    print(f"  Future (not started yet): {future}")
    print(f"  Ongoing (active now):     {ongoing}")
    print(f"  Past (already ended):     {past}")
    print("\n=> API contains active/future events." if future + ongoing > 0
          else "\n=> API appears to contain only past events.")
else:
    print("No start-date column detected — cannot classify.")

## 12. Pagination / record limits
Probe common paging schemes and compare returned counts. If the count changes with
`page`/`pageSize` (or the envelope carries a `total`), the endpoint is paginated.

In [ ]:
# a) Paging metadata in the envelope?
if isinstance(envelope, dict):
    meta = {k: v for k, v in envelope.items()
            if any(t in k.lower() for t in ["total", "count", "page", "size", "limit", "next"])
            and not isinstance(v, (list, dict))}
    print("Envelope paging metadata:", meta if meta else "(none found)")

# b) Does the count respond to paging params?
def count_for(params):
    try:
        r = requests.get(URL, headers=HEADERS, params=params, timeout=TIMEOUT)
        recs, _ = extract_records(r.json())
        return len(recs)
    except Exception as ex:
        return f"error: {ex}"

base_n = len(df)
print("Baseline (no params):", base_n)
for params in [{"page": 1, "pageSize": 10}, {"page": 2, "pageSize": 10},
               {"pageNumber": 1, "pageSize": 10}, {"limit": 10, "offset": 0},
               {"limit": 10, "offset": 10}, {"skip": 0, "take": 10},
               {"skip": 10, "take": 10}, {"pageSize": 1000}]:
    print(f"  {params} -> {count_for(params)} records")

print("\nInterpretation:")
print("- If page=1 and page=2 return DIFFERENT records -> paginated; iterate to get all.")
print("- If pageSize=1000 returns more than baseline -> baseline was capped by a default limit.")
print("- If every request returns the same count -> no pagination (single full payload).")

## Summary

Fill this in after running (values come from the cells above):

| # | Question | Finding |
|---|----------|---------|
| 1 | Number of events | _(Q1 / Q12 for true total)_ |
| 2 | Available fields | _(Q2 table)_ |
| 3 | Date coverage | _(Q3)_ |
| 4 | City coverage | _(Q4)_ |
| 5 | Categories / types | _(Q5)_ |
| 6 | Start / end dates | _(Q6)_ |
| 7 | Start / end times | _(Q6/7)_ |
| 8 | Family/male/female | _(Q8)_ |
| 9 | Missing values | _(Q9)_ |
| 10 | Duplicate events | _(Q10)_ |
| 11 | Active/future events | _(Q11)_ |
| 12 | Pagination / limits | _(Q12)_ |

**Then:** save the raw pull to `data/raw/events/` (e.g. `enjoy_events_<date>.json`) and
note the cleaning steps for `src/cleaning` (canonical city names to match other sources,
parse dates/times to ISO, normalise the audience/gender field).